# ALOHA：从导入到训练的端到端入口

本 Notebook 按顺序调用同目录的现有 Python 脚本，不复制导入、Prepare 或训练实现。

`原始数据 → Raw → 可选人工标注 → Prepared + Samples → 样本读取 → 可选训练 → 预测查看`

请先按 [运行环境](README.zh-CN.md#运行环境) 安装依赖并配置 PG/S3 环境变量，使用相同 Python 内核。从本示例目录或 LakeSoul 仓库根目录打开均可。导入需要本地 LeRobot v3 数据集；样本 Tensor 与训练需要 PyTorch。

默认只展示命令。修改配置后，将 `EXECUTE` 改为 `True`，重新运行配置单元格，再逐步执行。子进程使用当前内核的 Python、继承环境变量，失败会中止当前步骤。不要把密钥写入 Notebook。

已有 Prepared、只想查看数据时，请使用 [Prepared 快速查看](aloha_prepared_quickstart.ipynb)。

## 1. 统一配置

首次运行使用未占用的表名、存储路径和报告目录。复用已有 Raw 时设置 `IMPORT_RAW=False`，并把 `RAW_TABLE` 改为已有表名；该表必须带有导入元数据。只调整筛选策略时，保留 Raw 名称，为 Prepared、Samples 和报告目录换一个版本。

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

# 从示例目录或仓库根目录定位脚本；其他目录请直接填写绝对路径。
EXAMPLE_DIR = Path.cwd()
if not (EXAMPLE_DIR / "import_raw.py").is_file():
    EXAMPLE_DIR = EXAMPLE_DIR / "python/examples/embodied/aloha"
EXAMPLE_DIR = EXAMPLE_DIR.resolve()
if not (EXAMPLE_DIR / "import_raw.py").is_file():
    raise FileNotFoundError("请将 EXAMPLE_DIR 指向 python/examples/embodied/aloha")

EXECUTE = False                # True 才会实际读写 LakeSoul
IMPORT_RAW = True              # 已有 Raw 时改为 False
USE_ANNOTATIONS = False        # True 时在第 3 步完成标注后再继续
ANNOTATIONS_READY = False      # 标注发布成功后改为 True
RUN_TRAINING = False           # 可选训练，开启前需要 PyTorch

SOURCE = Path("/mnt/robot-data/aloha_static_thread_velcro")
NAMESPACE = "default"
STORAGE_ROOT = "s3://YOUR_BUCKET/embodied"
RAW_TABLE = "aloha_velcro_raw_v1"
PREPARED_TABLE = "aloha_velcro_prepared_v1"
SAMPLES_TABLE = "aloha_velcro_samples_v1"
REPORT_ROOT = Path("/tmp/aloha-pipeline-v1")
TRAIN_OUTPUT = REPORT_ROOT / "state_policy_output"

EPISODES = list(range(10))
CAMERAS = ["cam_high", "cam_low", "cam_left_wrist", "cam_right_wrist"]
BATCH_CAMERAS = ["cam_high"]
OBSERVATION_STEPS = 10
ACTION_STEPS = 10
ACTION_OFFSET = 0
STRIDE = 5
VALIDATION_RATIO = 0.3
TEST_RATIO = 0.1
SEED = 42
EPOCHS = 200
TRAIN_SAMPLES = 10000
VAL_SAMPLES = 10000


def run_script(name, arguments, *, new_output=None):
    """显示命令；实际执行时流式输出日志，并传播失败。"""
    command = [sys.executable, "-u", str(EXAMPLE_DIR / name), *map(str, arguments)]
    print(shlex.join(command), flush=True)
    if not EXECUTE:
        print("仅预览；将 EXECUTE 改为 True 后重新运行配置与所需步骤。")
        return
    if "YOUR_BUCKET" in STORAGE_ROOT:
        raise ValueError("请先填写 STORAGE_ROOT 中的实际桶名")
    if new_output is not None and Path(new_output).exists():
        raise FileExistsError(f"输出目录已存在，请换一个新目录：{new_output}")
    process = subprocess.Popen(
        command, cwd=EXAMPLE_DIR, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

## 2. 导入 Raw

调用 `import_raw.py` 写入数值记录与逐帧 JPEG，并发布后续 Prepare 需要的元数据。已有 Raw 时跳过，不会自动覆盖或删除已有表。

In [ ]:
if IMPORT_RAW:
    if EXECUTE and not (SOURCE / "meta/info.json").is_file():
        raise FileNotFoundError(f"找不到数据集元信息：{SOURCE / 'meta/info.json'}")
    run_script("import_raw.py", [
        "--source", SOURCE, "--namespace", NAMESPACE,
        "--episodes", ",".join(map(str, EPISODES)),
        "--cameras", ",".join(CAMERAS),
        "--raw-table", RAW_TABLE,
        "--raw-path", f"{STORAGE_ROOT.rstrip('/')}/{RAW_TABLE}",
        "--runner", "native", "--output", REPORT_ROOT / "import",
    ], new_output=REPORT_ROOT / "import")
else:
    print(f"复用已有 Raw：{NAMESPACE}.{RAW_TABLE}")

## 3. 可选：查看并发布人工标注

打开 [Raw 查看与标注](aloha_episode_viewer.ipynb)，填入下方打印的配置。
只浏览执行该 Notebook 第 1～4 步；标注执行第 1～3、5～6 步。发布成功后，在下面单元格中将 `ANNOTATIONS_READY` 改为 `True`，再继续 Prepare。

`USE_ANNOTATIONS=True` 会让下一步等待你明确完成发布，防止“运行全部”越过人工步骤。它只是本入口的流程开关；即使设置为 False，Prepare 仍会读取 Raw 已发布的最新标注，不代表忽略已有标注。切换轨迹前先发布草稿。

In [ ]:
print(f'NAMESPACE = {NAMESPACE!r}')
print(f'RAW_TABLE = {RAW_TABLE!r}')
print(f'EPISODE_ID = "ep{EPISODES[0]:06d}"')
# 仅在所需标注全部发布成功后改为 True。
ANNOTATIONS_READY = False

## 4. 生成 Prepared 和 Samples

调用 `prepare_dataset.py` 执行检查、筛选、Episode 划分和窗口索引构建。观察窗口为 t−9 到 t、动作窗口为 t 到 t+9（默认配置），候选锚点间隔 5 帧。报告写入本轮独立目录。修改标注后需使用新的输出表名、路径与报告目录重新执行。

In [ ]:
if EXECUTE and USE_ANNOTATIONS and not ANNOTATIONS_READY:
    raise RuntimeError("请先发布人工标注，并将 ANNOTATIONS_READY 设为 True")
run_script("prepare_dataset.py", [
    "--namespace", NAMESPACE, "--raw-table", RAW_TABLE,
    "--prepared-table", PREPARED_TABLE,
    "--prepared-path", f"{STORAGE_ROOT.rstrip('/')}/{PREPARED_TABLE}",
    "--samples-table", SAMPLES_TABLE,
    "--samples-path", f"{STORAGE_ROOT.rstrip('/')}/{SAMPLES_TABLE}",
    "--validation-ratio", VALIDATION_RATIO, "--test-ratio", TEST_RATIO,
    "--seed", SEED, "--observation-steps", OBSERVATION_STEPS,
    "--action-steps", ACTION_STEPS, "--action-offset", ACTION_OFFSET,
    "--stride", STRIDE, "--preview-samples", 1,
    "--runner", "native", "--output", REPORT_ROOT / "prepare",
], new_output=REPORT_ROOT / "prepare")

## 5. 回放 Prepared（可选）

打开 [Prepared 查看器](aloha_prepared_viewer.ipynb)，复制下方配置并依次执行其四步。选择 Episode 后点击加载，检查原始帧号、有效区间和保留画面。本入口不会在后台启动查看器。

In [ ]:
print(f'NAMESPACE = {NAMESPACE!r}')
print(f'PREPARED_TABLE = {PREPARED_TABLE!r}')
print(f'CAMERAS = {CAMERAS!r}')

## 6. 检查训练样本读取

调用 `read_training_batch.py` 读取一个训练样本。相机选择影响实际查询列；图片调整为宽 160、高 120，仅影响本次 Tensor，不修改表中 JPEG。这里不启动训练，也不是下一步训练的前置依赖。

In [ ]:
run_script("read_training_batch.py", [
    "--namespace", NAMESPACE, "--samples-table", SAMPLES_TABLE,
    "--split", "train", "--cameras", ",".join(BATCH_CAMERAS) or "none",
    "--batch-size", 1, "--image-size", 160, 120,
])

## 7. 可选：训练状态到动作的基线

将配置中的 `RUN_TRAINING=True` 后运行。脚本独立从 Samples 和 Prepared 读取状态与动作，不以图片为输入。样本数是读取上限，不会生成额外样本。训练输出采用本轮独立目录，避免混用不同训练的预测与索引。

In [ ]:
if RUN_TRAINING:
    run_script("train_state_policy.py", [
        "--namespace", NAMESPACE, "--samples-table", SAMPLES_TABLE,
        "--epochs", EPOCHS, "--train-samples", TRAIN_SAMPLES,
        "--val-samples", VAL_SAMPLES, "--seed", SEED,
        "--output", TRAIN_OUTPUT,
    ], new_output=TRAIN_OUTPUT)
else:
    print("未启用训练；需要时将 RUN_TRAINING 改为 True。")

## 8. 查看预测结果

训练成功后打开 [预测查看器](aloha_prediction_viewer.ipynb)，将其 `OUTPUT_DIR` 改为下面打印的路径（该查看器已导入 `Path`），重新运行加载单元格。该目录必须包含同一次训练的 `predictions.npz`、`samples.json` 和 `metrics.json`。查看器回读 Prepared 画面，展示真实动作与预测动作；画面并非模型执行视频，离线误差不等同于实机成功率。

In [ ]:
print(f'OUTPUT_DIR = Path({str(TRAIN_OUTPUT.resolve())!r})')
if not (TRAIN_OUTPUT / "predictions.npz").is_file():
    print("此目录尚无预测结果；请先成功完成训练。")